# 工具的应用案例

## 1、使用args_schema

举例1

In [ ]:
#模型初始化
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv(override=True)

model = init_chat_model(
    model="kimi-k2.6",
    model_provider="openai",
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url=os.getenv("MOONSHOT_BASE_URL")
)

In [ ]:
#定义工具
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel,Field
from langchain_core.tools import tool

class WeatherSchema(BaseModel):
     city : str = Field(default="江西",description="具体的城市名称")
     if_forecast : bool = Field(default=False,description="是否包含明天的天气")

@tool("get_weather_and_forecast",description="查询当日的天气，可以包含明天的天气预报",args_schema=WeatherSchema)
def get_weather(city : str,if_forecast:bool):
   res = f"{city}今天天气不错"
   if if_forecast:
       res += "\n明天下雨"
   return  res

# print(convert_to_openai_tool(get_weather))

In [6]:
from langchain_core.messages import HumanMessage

# 1、将工具绑定到模型上
model_with_tool = model.bind_tools([get_weather])

# 2、维护一个消息列表
messages = [HumanMessage("今天上海的天气怎么样？明天呢？")]

# 3、调用模型，得到响应：AIMessage
response = model_with_tool.invoke(messages)
messages.append(response)

# 4、获取响应中的tool_calls字段信息
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
       # 5、调用工具(因为大模型不能直接调用工具，所以这里我们主动让工具调用执行)
       tool_message = get_weather.invoke(tool_call)
       messages.append(tool_message)

# 6、调用模型,又得到一个AIMessage
final_response = model.invoke(messages)

# 7、再把得到的这个AIMessage添加到消息列表中
messages.append(final_response)

# 8、遍历消息列表
for mes in messages:
    mes.pretty_print()

================================ Human Message =================================

今天上海的天气怎么样？明天呢？
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (get_weather_and_forecast_0)
 Call ID: get_weather_and_forecast_0
  Args:
    city: 上海
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

上海今天天气不错
明天下雨
================================== Ai Message ==================================

上海今天的天气不错，适合外出活动。

不过明天会**下雨**，建议您出门记得带伞，并注意交通安全。


## 2、撰写docstring

In [11]:
#模型初始化
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv(override=True)

model = init_chat_model(
    model="kimi-k2.6",
    model_provider="openai",
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url=os.getenv("MOONSHOT_BASE_URL")
)

In [ ]:
#定义工具
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool

# class WeatherSchema(BaseModel):
#      city : str = Field(default="江西",description="具体的城市名称")
#      if_forecast : bool = Field(default=False,description="是否包含明天的天气")

@tool("get_weather_and_forecast",parse_docstring=True)
def get_weather(city : str = "上海",if_forecast:bool = False):
    """
    查询当日的天气，可以包含明天的天气预报

   Args:
       city : 城市名称
       if_forecast : 是否包含明天的天气
    """
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return  res

# print(convert_to_openai_tool(get_weather))

In [15]:
from langchain_core.messages import HumanMessage

# 1、将工具绑定到模型上
model_with_tool = model.bind_tools([get_weather])

# 2、维护一个消息列表
messages = [HumanMessage("今天上海的天气怎么样？明天呢？")]

# 3、调用模型，得到响应：AIMessage
response = model_with_tool.invoke(messages)
messages.append(response)

# 4、获取响应中的tool_calls字段信息
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
       # 5、调用工具(因为大模型不能直接调用工具，所以这里我们主动让工具调用执行)
       tool_message = get_weather.invoke(tool_call)
       messages.append(tool_message)

# 6、调用模型,又得到一个AIMessage
final_response = model.invoke(messages)

# 7、再把得到的这个AIMessage添加到消息列表中
messages.append(final_response)

# 8、遍历消息列表
for mes in messages:
    mes.pretty_print()

================================ Human Message =================================

今天上海的天气怎么样？明天呢？
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (get_weather_and_forecast_0)
 Call ID: get_weather_and_forecast_0
  Args:
    city: 上海
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

上海今天天气不错
明天下雨
================================== Ai Message ==================================

根据天气信息：

- **今天**：上海的天气不错，适合外出活动。
- **明天**：预计会**下雨**，出门记得带伞哦。

如果需要更详细的温度、风力等信息，可以查看具体的天气预报应用获取实时数据。


## 3、多工具的调用

In [19]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv(override=True)

model = init_chat_model(
    model="kimi-k2.6",
    model_provider="openai",
    api_key = os.getenv("MOONSHOT_API_KEY"),
    base_url = os.getenv("MOONSHOT_BASE_URL")
)


@tool(parse_docstring=True)
def get_stock_price(company: str,timeframe: str = "today"):
    """
    获取指定公司的股票价格信息

    Args:
        company : 公司名称(如：苹果公司,微软公司,谷歌公司)
        timeframe : 时间范围(today-今日,week-本周,month-本月)
    """
    #模拟股票数据
    mock_data = {
        "苹果公司":{"today":185.20,"week":183.50,"month":180.75},
        "微软公司":{"today":415.86,"week":412.30,"month":405.42},
        "谷歌公司":{"today":15.42,"week":15.20,"month":14.85}
    }

    if company in mock_data:
        price = mock_data[company].get(timeframe,"未知时间范围")
        return f"{company}{timeframe}价格:{price}美元"
    else:
        return f"未找到股票代码{company}的数据"




@tool(parse_docstring=True)
def search_news(company : str):
    """
    搜索指定公司的财经新闻

    Args:
        company:公司名称

    Returns:
        公司的财经新闻，每个新闻占一行
    """

    #模拟新闻数据
    mock_news = {
        "苹果公司":[
            "苹果发布新款iPhone,股价上涨3%",
            "苹果与欧盟达成反垄断和解协议",
            "苹果将在印度扩大生产规模"
        ],
         "微软公司":[
            "微软Azure云业务季度增长超预期",
            "微软完成对Nuance的收购",
            "微软推出新一代AI助手Copilot"
        ],
         "谷歌公司":[
             "谷歌发布新AI模型，性能提升20%",
            "谷歌与OpenAI合作，开发新的AI助手",
            "谷歌在欧洲展开AI研究项目"
        ],
    }

    new_list = mock_news.get(company,[f"未找到{company}的相关信息"])
    return "\n".join(new_list)

# rprint(convert_to_openai_tool(search_news))


#初始化模型并绑定工具
tools = [get_stock_price,search_news]
model_with_tool = model.bind_tools(tools)

message_list = []
human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻")
# human_message = HumanMessage(content="比较一下微软和苹果的股价")
# human_message = HumanMessage(content="腾讯最近有什么重大新闻？")
# human_message = HumanMessage(content="海水为什么是咸的？")
message_list.append(human_message)

while True:
    response = model_with_tool.invoke(message_list)
    message_list.append(response)

    if not response.tool_calls:
        print("没有工具调用，直接返回答案")
        break

    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_stock_price":
            stock_result = get_stock_price.invoke(tool_call)
            print("stock_result:", stock_result)
            message_list.append(stock_result)
        if tool_call["name"] == "search_news":
            new_result = search_news.invoke(tool_call)
            print("new_result",new_result)
            message_list.append(new_result)

for mes in message_list:
    mes.pretty_print()


stock_result: content='苹果公司today价格:185.2美元' name='get_stock_price' tool_call_id='get_stock_price_0'
new_result content='苹果发布新款iPhone,股价上涨3%\n苹果与欧盟达成反垄断和解协议\n苹果将在印度扩大生产规模' name='search_news' tool_call_id='search_news_1'
没有工具调用，直接返回答案
================================ Human Message =================================

苹果公司今天的股价是多少？最近有什么新闻
================================== Ai Message ==================================

我来帮您查询苹果公司今天的股价和最近的新闻。
Tool Calls:
  get_stock_price (get_stock_price_0)
 Call ID: get_stock_price_0
  Args:
    company: 苹果公司
    timeframe: today
  search_news (search_news_1)
 Call ID: search_news_1
  Args:
    company: 苹果公司
================================= Tool Message =================================
Name: get_stock_price

苹果公司today价格:185.2美元
================================= Tool Message =================================
Name: search_news

苹果发布新款iPhone,股价上涨3%
苹果与欧盟达成反垄断和解协议
苹果将在印度扩大生产规模
================================== Ai Message ==================================



## 4、多工具调用

In [30]:
from dotenv import load_dotenv
from langchain.chat_models.base import init_chat_model
import os
from langchain_core.tools import tool
from langchain.messages import HumanMessage
from builtins import str

load_dotenv(override=True)

model = init_chat_model(
    model="kimi-k2.6",
    model_provider="openai",
    api_key = os.getenv("MOONSHOT_API_KEY"),
    base_url = os.getenv("MOONSHOT_BASE_URL")
)

@tool(parse_docstring=True)
def get_weather(city:str):
    """
    获取城市的天气

    Args:
        city:城市名称
    """
    return f"{city}当天晴朗"

@tool(parse_docstring=True)
def get_news():
    """
    获取当日新闻
    """
    return "近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。"

model_with_tool = model.bind_tools([get_weather, get_news])

messages = [HumanMessage(content="今天杭州天气如何?今天新闻是什么?别瞎编")]

response = model_with_tool.invoke(messages)
messages.append(response)

for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_msg = get_weather.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    elif tool_call["name"] == "get_news":
        tool_msg = get_news.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    else:
        raise Exception("不存在的工具")

final_response = model.invoke(messages)
messages.append(final_response)

for mes in messages:
    mes.pretty_print()

content='杭州当天晴朗' name='get_weather' tool_call_id='get_weather_0'
content='近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。' name='get_news' tool_call_id='get_news_1'
================================ Human Message =================================

今天杭州天气如何?今天新闻是什么?别瞎编
================================== Ai Message ==================================

我来帮您查询杭州今天的天气和当日新闻。
Tool Calls:
  get_weather (get_weather_0)
 Call ID: get_weather_0
  Args:
    city: 杭州
  get_news (get_news_1)
 Call ID: get_news_1
  Args:
================================= Tool Message =================================
Name: get_weather

杭州当天晴朗
================================= Tool Message =================================
Name: get_news

近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。
================================== Ai Message ==================================

根据查询结果，为您如实汇报如下：

**杭州天气**：当天为**晴朗**天气。

**今日新闻**：近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。

*注：以上信息基于实时查询，未